## Cell 1 — Imports

Imports all required libraries: PyTorch, torchvision, albumentations, scikit-learn, OpenCV and standard utilities.

In [ ]:
import os
import sys
import random
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from pathlib import Path
from typing import Tuple, List, Dict, Optional, Any
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
from torchvision import transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Try different import methods for Swin-Base
try:
    from torchvision.models import swin_b, Swin_B_Weights
    print("\u2713 Swin-Base imported from torchvision.models")
except ImportError:
    try:
        import timm
        print("\u2713 Using timm library for Swin-Base")
        HAS_TIMM = True
    except ImportError:
        print("\u26a0 Warning: Neither torchvision Swin nor timm is available")
        print("Installing timm...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'timm'])
        import timm
        HAS_TIMM = True

warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")


## Cell 2 — Seed and Device Setup

Sets global random seeds for reproducibility and detects the available compute device.

In [ ]:
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(250)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = DEVICE  # lowercase alias used by downstream cells
print(f"\U0001f5a5\ufe0f  Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"\U0001f4ca GPU: {torch.cuda.get_device_name(0)}")
    print(f"\U0001f4be GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## Cell 3 — Configuration

Defines the `Config` class containing all hyperparameters — paths, image size, training settings, augmentation, FPN/GCA dimensions, and K-Fold settings.

In [ ]:
# CELL 3: CONFIGURATION PARAMETERS
# ============================================================================
class Config:
    """Central configuration for the Swin-Base multi-task training pipeline."""

    # ── Paths ──────────────────────────────────────────────────────────────────
    # Point DATA_ROOT at the raw (unsplit) dataset directory.
    # Expected structure:  DATA_ROOT / "Plant_Health" / image.jpg
    # e.g.  DATA_ROOT / "Guava_Healthy" / img001.jpg
    DATA_ROOT = Path("/workspace/Dataset_raw")   # <-- update this path
    OUTPUT_DIR = Path("outputs")
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    LOG_DIR = OUTPUT_DIR / "logs"

    # ── Dataset parameters ─────────────────────────────────────────────────────
    PLANTS  = ['Guava', 'Mango', 'Papaya']
    CLASSES = ['Healthy', 'Anthracnose']
    SPECIES_CLASSES = ['guava', 'mango', 'papaya']
    HEALTH_CLASSES  = ['healthy', 'anthracnose']
    NUM_SPECIES = 3
    NUM_HEALTH  = 2

    # ── Data split ratios ──────────────────────────────────────────────────────
    TRAIN_RATIO = 0.70
    VAL_RATIO   = 0.15
    TEST_RATIO  = 0.15

    # ── Image parameters ───────────────────────────────────────────────────────────
    IMG_SIZE = 224
    IMG_MEAN = [0.485, 0.456, 0.406]
    IMG_STD  = [0.229, 0.224, 0.225]

    # ── Training parameters ──────────────────────────────────────────────────────
    BATCH_SIZE    = 16
    NUM_WORKERS   = 0
    EPOCHS        = 50
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY  = 5e-4
    DROPOUT       = 0.3

    # Scheduler
    MIN_LR = 1e-7

    # Early stopping
    PATIENCE = 5

    # K-Fold
    N_FOLDS    = 5
    SEED       = 250
    STATE_FILE = "kfold_state.json"

    # Loss
    LOSS_WEIGHT_HEALTH = 1.0

    # AMP
    USE_AMP = True

    # Model architecture
    FPN_CHANNELS  = 256
    GCA_REDUCTION = 16
    PRETRAINED    = False

    # Augmentation
    AUG_PROB = 0.5

    @classmethod
    def create_dirs(cls):
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        cls.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        cls.LOG_DIR.mkdir(parents=True, exist_ok=True)
        print("\u2705 Output directories created")

Config.create_dirs()

# ── Flat aliases (used by downstream cells) ───────────────────────────────────────────────
DATA_ROOT          = Config.DATA_ROOT
IMG_SIZE           = Config.IMG_SIZE
BATCH_SIZE         = Config.BATCH_SIZE
NUM_WORKERS        = Config.NUM_WORKERS
EPOCHS             = Config.EPOCHS
LR                 = Config.LEARNING_RATE
WEIGHT_DECAY       = Config.WEIGHT_DECAY
DROPOUT            = Config.DROPOUT
FPN_CHANNELS       = Config.FPN_CHANNELS
GCA_REDUCTION      = Config.GCA_REDUCTION
SEED               = Config.SEED
LOSS_WEIGHT_HEALTH = Config.LOSS_WEIGHT_HEALTH
USE_AMP            = Config.USE_AMP
PATIENCE           = Config.PATIENCE
N_FOLDS            = Config.N_FOLDS
STATE_FILE         = Config.STATE_FILE
NUM_SPECIES        = Config.NUM_SPECIES
NUM_HEALTH         = Config.NUM_HEALTH
PRETRAINED         = Config.PRETRAINED
SPECIES_MAP        = {"guava": 0, "mango": 1, "papaya": 2}
HEALTH_MAP         = {"healthy": 0, "anthracnose": 1}

print("\n\U0001f4cb Configuration Summary:")
print(f"   - Data Root    : {Config.DATA_ROOT}")
print(f"   - Image Size   : {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"   - Batch Size   : {Config.BATCH_SIZE}")
print(f"   - Learning Rate: {Config.LEARNING_RATE}")
print(f"   - Epochs       : {Config.EPOCHS}")
print(f"   - Split        : {Config.TRAIN_RATIO}/{Config.VAL_RATIO}/{Config.TEST_RATIO} (train/val/test)")
print(f"   - N Folds      : {Config.N_FOLDS}")
print(f"   - FPN Channels : {Config.FPN_CHANNELS}")
print(f"   - GCA Reduction: {Config.GCA_REDUCTION}")


## Cell 4 — Dataset Loader

Defines `DatasetLoader` for scanning the raw dataset directory and `FlatSampleDataset` for per-fold train/val loaders.

In [ ]:
class DatasetLoader:
    """Load and organise the raw dataset into a DataFrame with multi-task labels.

    Expected directory structure (unsplit raw dataset):
        DATA_ROOT / Plant_Health / image.jpg
    e.g.
        DATA_ROOT / Guava_Healthy     / img001.jpg
        DATA_ROOT / Guava_Anthracnose / img002.jpg
        DATA_ROOT / Mango_Healthy     / img003.jpg
        ...
    """

    VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

    def __init__(self, data_root: Path, plants: List[str], classes: List[str]):
        self.data_root = Path(data_root)
        self.plants    = plants
        self.classes   = classes

    def load_dataset(self) -> pd.DataFrame:
        """Scan all plant/class folders and return a DataFrame.

        Columns: image_path, plant, class, species_id, health_id, label
        where  label = health_id  (0 = Healthy, 1 = Anthracnose)
        """
        data = []
        for plant in self.plants:
            for cls in self.classes:
                folder_name = f"{plant}_{cls}"
                folder_path = self.data_root / folder_name

                if not folder_path.exists():
                    print(f"\u26a0\ufe0f  Warning: {folder_path} not found \u2014 skipping")
                    continue

                imgs = [str(p) for p in folder_path.iterdir()
                        if p.suffix.lower() in self.VALID_EXTS]

                sp_id = SPECIES_MAP[plant.lower()]
                he_id = HEALTH_MAP[cls.lower()]

                for img_path in imgs:
                    data.append({
                        'image_path': img_path,
                        'plant':      plant,
                        'class':      cls,
                        'species_id': sp_id,
                        'health_id':  he_id,
                        'label':      he_id,
                        'folder':     folder_name,
                    })

        df = pd.DataFrame(data)
        print(f"\u2705 Loaded {len(df)} images from {df['folder'].nunique()} folders")
        return df

    @staticmethod
    def compute_class_weights(labels: np.ndarray) -> torch.Tensor:
        """Compute class weights for imbalanced datasets."""
        cw = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
        return torch.tensor(cw, dtype=torch.float32)

    @staticmethod
    def get_sample_weights(labels: np.ndarray) -> np.ndarray:
        """Per-sample weights for WeightedRandomSampler."""
        class_counts = Counter(labels)
        total = len(labels)
        cw = {cls: total / cnt for cls, cnt in class_counts.items()}
        return np.array([cw[lbl] for lbl in labels])


class FlatSampleDataset(Dataset):
    """Wraps an explicit list of (path, species_id, health_id) tuples.
    Applies albumentations transforms after converting PIL to numpy.
    Used to build per-fold train/val loaders.
    """

    def __init__(self, samples: List[Tuple[str, int, int]], transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, sp_id, he_id = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        if self.transform is not None:
            img_np = np.array(img)                        # PIL \u2192 numpy HWC uint8
            img = self.transform(image=img_np)['image']   # albumentations call
        else:
            img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)


print("\u2705 DatasetLoader and FlatSampleDataset defined")


## Cell 5 — Stratified Split

Loads the raw dataset, performs a 70/15/15 stratified split, and builds `all_samples` + `strat_labels` for K-Fold cross-validation.

In [ ]:
# ── Helper: stratified split (adapted from ResNet50) ─────────────────────────────
from sklearn.model_selection import train_test_split

def stratified_split(
        df: pd.DataFrame,
        train_ratio: float = 0.70,
        val_ratio:   float = 0.15,
        test_ratio:  float = 0.15,
        random_state: int  = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Stratified split maintaining species\u00d7health balance across all three sets.
    Stratifies on plant+class combination so every combo is represented equally.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Ratios must sum to 1.0"

    df = df.copy()
    df['strat_col'] = df['plant'] + '_' + df['class']

    # Step 1: carve out the test set
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_ratio,
        stratify=df['strat_col'],
        random_state=random_state
    )

    # Step 2: split the remainder into train / val
    relative_val = val_ratio / (train_ratio + val_ratio)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val,
        stratify=train_val_df['strat_col'],
        random_state=random_state
    )

    for split_df in [train_df, val_df, test_df]:
        split_df.drop('strat_col', axis=1, inplace=True)

    return (train_df.reset_index(drop=True),
            val_df.reset_index(drop=True),
            test_df.reset_index(drop=True))


# ── 1. Load raw dataset ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("Loading raw dataset...")
print("="*80)

loader = DatasetLoader(Config.DATA_ROOT, Config.PLANTS, Config.CLASSES)
df = loader.load_dataset()

print("\n\U0001f4ca Dataset Statistics:")
print(df.groupby(['plant', 'class']).size().unstack(fill_value=0))
print("\n\U0001f4c8 Overall Class Distribution:")
print(df['class'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='class', ax=axes[0], palette='viridis')
axes[0].set_title('Overall Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

df_melted = df.groupby(['plant', 'class']).size().reset_index(name='count')
sns.barplot(data=df_melted, x='plant', y='count', hue='class',
            ax=axes[1], palette='viridis')
axes[1].set_title('Class Distribution by Plant', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Plant Type'); axes[1].set_ylabel('Count')
axes[1].legend(title='Class')

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Stratified split (70 / 15 / 15) ───────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("Performing stratified split (70 / 15 / 15)...")
print("="*80)

train_df, val_df, test_df = stratified_split(
    df, Config.TRAIN_RATIO, Config.VAL_RATIO, Config.TEST_RATIO,
    random_state=Config.SEED
)

print("\n\U0001f4ca Split Statistics:")
print(f"   Training  : {len(train_df):5d} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Validation: {len(val_df):5d} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Test      : {len(test_df):5d} samples ({len(test_df)/len(df)*100:.1f}%)")

print("\n\U0001f4c8 Class Distribution in Splits:")
for name, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = sdf['class'].value_counts()
    print(f"   {name:6s}: Healthy={dist.get('Healthy', 0):4d}, "
          f"Anthracnose={dist.get('Anthracnose', 0):4d}")

train_df.to_csv(Config.OUTPUT_DIR / 'train_split.csv', index=False)
val_df.to_csv(Config.OUTPUT_DIR   / 'val_split.csv',   index=False)
test_df.to_csv(Config.OUTPUT_DIR  / 'test_split.csv',  index=False)
print("\n\u2705 Splits saved to output directory")

# ── 3. Build K-Fold pool and fixed test set ─────────────────────────────────────────────────────
def _df_to_samples(sdf: pd.DataFrame) -> List[Tuple[str, int, int]]:
    return [(row.image_path, row.species_id, row.health_id)
            for _, row in sdf.iterrows()]

all_samples      = _df_to_samples(pd.concat([train_df, val_df], ignore_index=True))
test_samples_raw = _df_to_samples(test_df)

strat_labels = [sp * NUM_HEALTH + he for (_, sp, he) in all_samples]

health_labels_pool    = np.array([he for (_, _, he) in all_samples])
health_class_weights  = loader.compute_class_weights(health_labels_pool)
health_sample_weights = loader.get_sample_weights(health_labels_pool)
print(f"\n\u2696\ufe0f  Health Class Weights: {health_class_weights.tolist()}")

print(f"\n{'='*80}")
print(f"K-Fold Dataset Summary:")
print(f"  Train+Val pool : {len(all_samples):5d} images (split per fold)")
print(f"  Test (fixed)   : {len(test_samples_raw):5d} images")
print(f"  K-Folds        : {N_FOLDS}")
print(f"  Approx per fold \u2014 train: {int(len(all_samples)*(N_FOLDS-1)/N_FOLDS)}, "
      f"val: {int(len(all_samples)/N_FOLDS)}")
print("="*80)


## Cell 6 — Augmentation Pipeline

Defines `AugmentationPipeline` using albumentations with 9 transforms for training and simple resize+normalize for validation. Sets `train_tf` and `eval_tf` used by `FlatSampleDataset`.

In [ ]:
class AugmentationPipeline:
    """Comprehensive augmentation pipeline using albumentations."""

    @staticmethod
    def get_train_transforms(img_size: int = 224, aug_prob: float = 0.5) -> A.Compose:
        return A.Compose([
            A.Rotate(limit=30, p=aug_prob, border_mode=cv2.BORDER_REFLECT_101),
            A.HorizontalFlip(p=aug_prob),
            A.VerticalFlip(p=aug_prob * 0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=aug_prob),
            A.RandomResizedCrop(
                size=(img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=aug_prob
            ),
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.1, p=aug_prob),
            A.GaussianBlur(blur_limit=(3, 7), p=aug_prob * 0.3),
            A.RandomShadow(
                shadow_roi=(0, 0.5, 1, 1),
                num_shadows_lower=1, num_shadows_upper=2,
                shadow_dimension=5, p=aug_prob * 0.3
            ),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=aug_prob * 0.3),
            A.CoarseDropout(
                max_holes=8,
                max_height=img_size // 8, max_width=img_size // 8,
                min_holes=1,
                min_height=img_size // 16, min_width=img_size // 16,
                fill_value=0, p=aug_prob * 0.3
            ),
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    @staticmethod
    def get_val_transforms(img_size: int = 224) -> A.Compose:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    @staticmethod
    def get_visualization_transforms(img_size: int = 224) -> A.Compose:
        return A.Compose([
            A.Rotate(limit=30, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.1, p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
            A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.3),
            A.Resize(img_size, img_size),
        ])

train_tf  = AugmentationPipeline.get_train_transforms(Config.IMG_SIZE, Config.AUG_PROB)
eval_tf   = AugmentationPipeline.get_val_transforms(Config.IMG_SIZE)
vis_transforms = AugmentationPipeline.get_visualization_transforms(Config.IMG_SIZE)

print("\u2705 Augmentation pipelines created")
print(f"   train_tf : {len(train_tf)}  transforms (with augmentation)")
print(f"   eval_tf  : {len(eval_tf)} transforms (resize + normalize only)")


## Cell 7 — Visualise Augmentations

Displays a grid of augmented versions of sample images from each class.

In [ ]:
def visualize_augmentations(
        samples: List[Tuple[str, int, int]],
        vis_transform: A.Compose,
        num_samples: int = 3,
        num_augmentations: int = 5,
        save_path: Optional[Path] = None):
    """Visualise augmentation effects on sample images from each species/health combo."""
    REVERSE_SP = {v: k for k, v in SPECIES_MAP.items()}
    REVERSE_HE = {v: k for k, v in HEALTH_MAP.items()}

    from collections import defaultdict
    groups = defaultdict(list)
    for path, sp, he in samples:
        groups[(sp, he)].append(path)

    class_keys = sorted(groups.keys())
    total_rows = len(class_keys) * num_samples
    fig, axes = plt.subplots(
        total_rows, num_augmentations + 1,
        figsize=(3 * (num_augmentations + 1), 3 * total_rows)
    )
    if total_rows == 1:
        axes = axes[np.newaxis, :]

    row_idx = 0
    for (sp, he) in class_keys:
        paths = groups[(sp, he)]
        chosen = random.sample(paths, min(num_samples, len(paths)))
        label_str = f"{REVERSE_SP[sp]} / {REVERSE_HE[he]}"
        for img_path in chosen:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE))
            axes[row_idx, 0].imshow(img)
            axes[row_idx, 0].set_title(f'Original\n({label_str})', fontsize=9)
            axes[row_idx, 0].axis('off')
            for aug_idx in range(num_augmentations):
                augmented = vis_transform(image=img)['image']
                axes[row_idx, aug_idx + 1].imshow(augmented)
                axes[row_idx, aug_idx + 1].set_title(f'Aug {aug_idx + 1}', fontsize=9)
                axes[row_idx, aug_idx + 1].axis('off')
            row_idx += 1

    plt.suptitle('Augmentation Pipeline Visualisation\n(5 Augmentations per Sample)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\u2705 Visualisation saved to {save_path}")
    plt.show()


print("\U0001f504 Visualising augmentation effects...")
visualize_augmentations(
    all_samples, vis_transforms,
    num_samples=2, num_augmentations=5,
    save_path=Config.OUTPUT_DIR / 'augmentation_visualization.png'
)


## Cell 8 — Demonstrate Augmentations

Shows each of the key augmentations applied individually to a sample image.

In [ ]:
def demonstrate_individual_augmentations(image_path: str, save_path: Optional[Path] = None):
    """Demonstrate each of the 5 main augmentations individually."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE))

    augmentations = {
        'Original': A.Compose([A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)]),
        '1. Rotation (\u00b130\u00b0)': A.Compose([
            A.Rotate(limit=30, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
        '2. H/V Flip': A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
        '3. Brightness/Contrast': A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
        '4. Random Crop': A.Compose([
            A.RandomResizedCrop(size=(Config.IMG_SIZE, Config.IMG_SIZE), scale=(0.7, 0.9), p=1.0)
        ]),
        '5. Color Jitter': A.Compose([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.15, p=1.0),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
    }

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for idx, (name, transform) in enumerate(augmentations.items()):
        augmented = transform(image=img)['image']
        axes[idx].imshow(augmented)
        axes[idx].set_title(name, fontsize=11, fontweight='bold')
        axes[idx].axis('off')

    plt.suptitle('5 Key Augmentations Demonstrated', fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


sample_img_path = all_samples[0][0]
demonstrate_individual_augmentations(
    sample_img_path,
    save_path=Config.OUTPUT_DIR / 'individual_augmentations.png'
)


## Cell 9 — Anthracnose Dataset

Defines `AnthracnoseDataset`, the PyTorch `Dataset` subclass that loads images and returns `(image, species_label, health_label)` for multi-task learning.

In [ ]:
class AnthracnoseDataset(Dataset):
    """
    Custom dataset for Anthracnose multi-task classification.
    Returns (image_tensor, species_label, health_label).
    Accepts albumentations transforms — applies them after PIL→numpy conversion.
    """

    def __init__(self,
                 samples: List[Tuple[str, int, int]],
                 transform: Optional[A.Compose] = None):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, sp_id, he_id = self.samples[idx]
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)['image']
        else:
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)


print("\u2705 AnthracnoseDataset (multi-task) defined")


## Cell 10 — Create Dataloaders

Defines `create_dataloaders()` and creates the fixed `test_loader` used across all folds.

In [ ]:
def create_dataloaders(
        train_samples: List[Tuple[str, int, int]],
        val_samples:   List[Tuple[str, int, int]],
        test_samples:  List[Tuple[str, int, int]],
        batch_size: int,
        sample_weights: Optional[np.ndarray] = None,
        num_workers: int = 0
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Create data loaders; optionally uses WeightedRandomSampler for training."""
    train_ds = AnthracnoseDataset(train_samples, transform=train_tf)
    val_ds   = AnthracnoseDataset(val_samples,   transform=eval_tf)
    test_ds  = AnthracnoseDataset(test_samples,  transform=eval_tf)

    if sample_weights is not None:
        sampler = WeightedRandomSampler(
            weights=sample_weights, num_samples=len(sample_weights), replacement=True
        )
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, sampler=sampler,
            num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"), drop_last=True
        )
    else:
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True,
            num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"), drop_last=True
        )

    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=(DEVICE.type == "cuda")
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=(DEVICE.type == "cuda")
    )
    return train_loader, val_loader, test_loader


# Fixed test loader — used in every fold's evaluation and best-fold eval
_, _, test_loader = create_dataloaders(
    train_samples=all_samples,
    val_samples=all_samples,
    test_samples=test_samples_raw,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

print(f"\u2705 test_loader created: {len(test_loader)} batches ({len(test_samples_raw)} images)")
print(f"\u2705 all_samples pool    : {len(all_samples)} images")
print(f"\u2705 strat_labels        : {len(strat_labels)} entries")
print(f"\u2705 train_tf / eval_tf  : albumentations pipelines ready")


## MODEL DEFINITION

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Global Context Attention (identical to ResNet50 implementation)
# ─────────────────────────────────────────────────────────────────────────────

class GlobalContextAttention(nn.Module):
    """
    Global Context Attention Block for refining feature representations.

    This module captures long-range dependencies and global context,
    which is crucial for identifying disease patterns across the entire leaf.

    Architecture:
    - Global Average Pooling to capture global context
    - Channel attention via squeeze-excitation mechanism
    - Spatial attention for locating disease regions
    - Feature refinement through residual connection
    """

    def __init__(self, in_channels: int, reduction: int = 16):
        """
        Args:
            in_channels: Number of input channels
            reduction: Reduction ratio for squeeze-excitation
        """
        super().__init__()

        # Channel attention (Squeeze-Excitation)
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

        # Spatial attention
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, 1, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        # Global context modeling
        self.global_context = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        # Layer normalization for stability
        self.norm = nn.LayerNorm([in_channels])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of GCA block.

        Args:
            x: Input tensor of shape (B, C, H, W)

        Returns:
            Refined feature tensor of shape (B, C, H, W)
        """
        batch_size, channels, height, width = x.shape

        # Channel attention
        channel_weights = self.channel_attention(x)
        channel_weights = channel_weights.view(batch_size, channels, 1, 1)
        channel_refined = x * channel_weights

        # Spatial attention
        spatial_weights = self.spatial_attention(x)
        spatial_refined = x * spatial_weights

        # Combine channel and spatial attention
        combined = channel_refined + spatial_refined

        # Global context
        global_ctx = self.global_context(combined)

        # Residual connection
        output = x + global_ctx

        return output


class MultiHeadGCA(nn.Module):
    """
    Multi-head Global Context Attention for richer feature refinement.
    Uses multiple attention heads to capture different aspects of the disease.
    """

    def __init__(self, in_channels: int, num_heads: int = 4, reduction: int = 16):
        super().__init__()

        assert in_channels % num_heads == 0, "in_channels must be divisible by num_heads"

        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads

        self.heads = nn.ModuleList([
            GlobalContextAttention(self.head_dim, reduction)
            for _ in range(num_heads)
        ])

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, channels, height, width = x.shape

        # Split into heads
        x_split = x.chunk(self.num_heads, dim=1)

        # Apply attention to each head
        attended = [head(x_i) for head, x_i in zip(self.heads, x_split)]

        # Concatenate and fuse
        output = torch.cat(attended, dim=1)
        output = self.fusion(output)

        return output


# ─────────────────────────────────────────────────────────────────────────────
# Swin-Base FPN Backbone
# ─────────────────────────────────────────────────────────────────────────────

class SwinBaseFPNBackbone(nn.Module):
    """
    Swin-Base backbone with Feature Pyramid Network.

    Swin-Base is hierarchical — it produces 4 genuine multi-scale spatial maps:
      Stage 1: (B, 128,  56x56)
      Stage 2: (B, 256,  28x28)
      Stage 3: (B, 512,  14x14)
      Stage 4: (B, 1024,  7x7)

    Hooks capture each stage output. torchvision Swin outputs (B, H, W, C)
    NHWC tensors which are permuted to (B, C, H, W) before the FPN.
    The top-down FPN pathway provides genuine multi-scale feature fusion.
    """

    # Swin-Base channel dims per stage
    STAGE_DIMS = [128, 256, 512, 1024]

    def __init__(self, swin_model, fpn_channels: int = 256):
        super().__init__()
        self.swin = swin_model
        self.fpn_channels = fpn_channels
        self._feature_maps: Dict[int, torch.Tensor] = {}
        self._hooks = []
        self._register_hooks()

        # Lateral connections — each stage has a different channel count
        self.lateral1 = nn.Conv2d(self.STAGE_DIMS[0], fpn_channels, 1)
        self.lateral2 = nn.Conv2d(self.STAGE_DIMS[1], fpn_channels, 1)
        self.lateral3 = nn.Conv2d(self.STAGE_DIMS[2], fpn_channels, 1)
        self.lateral4 = nn.Conv2d(self.STAGE_DIMS[3], fpn_channels, 1)

        # Smooth layers
        self.smooth1 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)
        self.smooth2 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)
        self.smooth3 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)
        self.smooth4 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)

        self.out_channels = fpn_channels

    def _get_stages(self):
        """Return the 4 stage modules for both torchvision and timm Swin."""
        if hasattr(self.swin, 'features'):
            # torchvision: features[1], [3], [5], [7] are the 4 stage block sequences
            return [self.swin.features[1], self.swin.features[3],
                    self.swin.features[5], self.swin.features[7]]
        elif hasattr(self.swin, 'layers'):
            # timm
            return list(self.swin.layers)
        return None

    def _register_hooks(self):
        """Register forward hooks on each of the 4 Swin stages."""
        stages = self._get_stages()
        if stages is None:
            raise RuntimeError("Cannot find Swin stages in model.")
        for stage_idx, stage in enumerate(stages):
            hook = stage.register_forward_hook(
                lambda m, inp, out, idx=stage_idx: self._feature_maps.update({idx: out})
            )
            self._hooks.append(hook)

    def _to_nchw(self, feat: torch.Tensor) -> torch.Tensor:
        """
        Convert NHWC tensor (B, H, W, C) to NCHW (B, C, H, W).
        torchvision Swin stage outputs are NHWC; detected when last dim > dim 1.
        """
        if feat.dim() == 4 and feat.shape[-1] > feat.shape[1]:
            return feat.permute(0, 3, 1, 2).contiguous()
        return feat

    def _upsample_add(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        _, _, h, w = y.size()
        return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False) + y

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        self._feature_maps = {}

        # Forward through Swin — hooks capture each stage output
        _ = self.swin(x)

        # Convert NHWC → NCHW if needed (torchvision Swin outputs NHWC)
        c1 = self._to_nchw(self._feature_maps[0])  # stage 1: 56x56, 128ch
        c2 = self._to_nchw(self._feature_maps[1])  # stage 2: 28x28, 256ch
        c3 = self._to_nchw(self._feature_maps[2])  # stage 3: 14x14, 512ch
        c4 = self._to_nchw(self._feature_maps[3])  # stage 4:  7x7, 1024ch

        # FPN top-down pathway (genuine multi-scale for Swin)
        p4 = self.lateral4(c4)
        p3 = self._upsample_add(p4, self.lateral3(c3))
        p2 = self._upsample_add(p3, self.lateral2(c2))
        p1 = self._upsample_add(p2, self.lateral1(c1))

        # Smooth
        p4 = self.smooth4(p4)
        p3 = self.smooth3(p3)
        p2 = self.smooth2(p2)
        p1 = self.smooth1(p1)

        return {'p1': p1, 'p2': p2, 'p3': p3, 'p4': p4}


# ─────────────────────────────────────────────────────────────────────────────
# Multi-Task Swin-Base with FPN + GCA
# ─────────────────────────────────────────────────────────────────────────────

class MultiTaskSwinBase(nn.Module):
    def __init__(self, num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                 pretrained=PRETRAINED, dropout=DROPOUT,
                 fpn_channels=FPN_CHANNELS, gca_reduction=GCA_REDUCTION):
        super().__init__()

        # ── Load Swin-Base backbone ───────────────────────────────────────────
        try:
            from torchvision.models import swin_b, Swin_B_Weights
            if pretrained:
                weights = Swin_B_Weights.IMAGENET1K_V1
                swin_model = swin_b(weights=weights)
            else:
                swin_model = swin_b(weights=None)
            swin_model.head = nn.Identity()
            print("\u2713 Using Swin-Base from torchvision")

        except (ImportError, AttributeError):
            import timm
            if pretrained:
                model_name = 'swin_base_patch4_window7_224.ms_in22k_ft_in1k'
            else:
                model_name = 'swin_base_patch4_window7_224'
            swin_model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
            print(f"\u2713 Using Swin-Base from timm library (model: {model_name})")

        # ── FPN Backbone (wraps Swin-Base with Feature Pyramid Network) ───────
        self.backbone = SwinBaseFPNBackbone(swin_model, fpn_channels=fpn_channels)

        # ── Multi-Head GCA applied to each FPN level ──────────────────────────
        self.gca = MultiHeadGCA(fpn_channels, num_heads=4, reduction=gca_reduction)

        # ── Feature fusion (4 FPN levels → fpn_channels) ─────────────────────
        self.feature_fusion = nn.Sequential(
            nn.Conv2d(fpn_channels * 4, fpn_channels * 2, kernel_size=1, bias=False),
            nn.BatchNorm2d(fpn_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(fpn_channels * 2, fpn_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(fpn_channels),
            nn.ReLU(inplace=True)
        )

        # ── Global pooling ────────────────────────────────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # ── Multi-task classification heads ───────────────────────────────────
        self.dropout = nn.Dropout(dropout)
        self.head_species = nn.Linear(fpn_channels, num_species)
        self.head_health  = nn.Linear(fpn_channels, num_health)

        # ── Initialise new (non-pretrained) layers ────────────────────────────
        self._init_new_weights()

    def _init_new_weights(self):
        """Initialise weights for all newly added layers (FPN, GCA, heads)."""
        new_conv_layers = [
            self.backbone.lateral1, self.backbone.lateral2,
            self.backbone.lateral3, self.backbone.lateral4,
            self.backbone.smooth1,  self.backbone.smooth2,
            self.backbone.smooth3,  self.backbone.smooth4,
        ]
        for m in new_conv_layers:
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        for m in self.feature_fusion.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        for linear in [self.head_species, self.head_health]:
            nn.init.xavier_normal_(linear.weight)
            nn.init.constant_(linear.bias, 0)

    def forward(self, x: torch.Tensor):
        # Extract multi-scale features via Swin-Base FPN
        features = self.backbone(x)

        # Apply GCA to each FPN level
        p1_att = self.gca(features['p1'])
        p2_att = self.gca(features['p2'])
        p3_att = self.gca(features['p3'])
        p4_att = self.gca(features['p4'])

        # Resize all FPN levels to match p4 spatial size (7x7), then concatenate
        target_size = p4_att.shape[-2:]
        p1_resized = F.adaptive_avg_pool2d(p1_att, target_size)
        p2_resized = F.adaptive_avg_pool2d(p2_att, target_size)
        p3_resized = F.adaptive_avg_pool2d(p3_att, target_size)

        multi_scale = torch.cat([p1_resized, p2_resized, p3_resized, p4_att], dim=1)
        fused = self.feature_fusion(multi_scale)

        # Global pooling → multi-task heads
        pooled = self.global_pool(fused).flatten(1)
        pooled = self.dropout(pooled)

        logits_species = self.head_species(pooled)
        logits_health  = self.head_health(pooled)

        return logits_species, logits_health

print("\u2713 Model class defined (instantiated fresh per fold)")


##  OPTIMIZER, SCHEDULER, LOSS FUNCTIONS

In [ ]:
criterion_species = nn.CrossEntropyLoss()
criterion_health  = nn.CrossEntropyLoss()
# scaler is reassigned as a global inside train_fold() for each fold
scaler = None
print("✓ Loss functions initialized")

## TRAINING UTILITIES

In [ ]:
def accuracy(logits, targets):
    """Calculate accuracy from logits and targets"""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

def run_epoch(loader, model, optimizer=None, train=True, epoch=0):
    """Run one epoch of training or evaluation"""
    if train:
        model.train()
    else:
        model.eval()
    
    running = {
        "loss": 0.0,
        "acc_species": 0.0,
        "acc_health": 0.0,
        "acc_both": 0.0,
        "n": 0
    }
    total_batches = len(loader)
    
    for batch_idx, (imgs, y_species, y_health) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        y_species = y_species.to(device, non_blocking=True)
        y_health = y_health.to(device, non_blocking=True)
        
        with torch.set_grad_enabled(train):
            if USE_AMP and device.type == "cuda":
                with torch.amp.autocast('cuda'):
                    logits_species, logits_health = model(imgs)
                    loss = criterion_species(logits_species, y_species) + \
                           LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
            else:
                logits_species, logits_health = model(imgs)
                loss = criterion_species(logits_species, y_species) + \
                       LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
        
        if train:
            optimizer.zero_grad(set_to_none=True)
            if USE_AMP and device.type == "cuda":
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        
        # Calculate accuracies
        acc_sp = accuracy(logits_species, y_species)
        acc_he = accuracy(logits_health, y_health)
        preds_sp = logits_species.argmax(1)
        preds_he = logits_health.argmax(1)
        acc_both = ((preds_sp == y_species) & (preds_he == y_health)).float().mean().item()
        
        # Update running statistics
        bs = imgs.size(0)
        running["loss"] += loss.item() * bs
        running["acc_species"] += acc_sp * bs
        running["acc_health"] += acc_he * bs
        running["acc_both"] += acc_both * bs
        running["n"] += bs
        
        # Print progress
        if (batch_idx + 1) % max(1, total_batches // 10) == 0 or (batch_idx + 1) == total_batches:
            avg_loss = running["loss"] / running["n"]
            avg_sp = running["acc_species"] / running["n"]
            avg_he = running["acc_health"] / running["n"]
            avg_both = running["acc_both"] / running["n"]
            print(f"  [{batch_idx + 1}/{total_batches}] loss: {avg_loss:.4f}, "
                  f"sp: {avg_sp:.3f}, he: {avg_he:.3f}, both: {avg_both:.3f}")
    
    # Calculate averages
    for k in ["loss", "acc_species", "acc_health", "acc_both"]:
        running[k] /= max(1, running["n"])
    
    return running

## TRAINING LOOP

In [ ]:
import glob as _glob
import os

def find_latest_epoch_ckpt(fold_idx):
    """Return the fold_{fold_idx}_epoch_*.pt with the highest epoch number, or None."""
    files = _glob.glob(f"fold_{fold_idx}_epoch_*.pt")
    if not files:
        return None
    def extract_epoch(fn):
        try:
            return int(fn.split("_epoch_")[1].replace(".pt", ""))
        except Exception:
            return -1
    files.sort(key=extract_epoch)
    return files[-1]


def train_fold(fold_idx, train_samples, val_samples):
    """Train one fold with full checkpoint-resume support."""
    global scaler

    torch.manual_seed(SEED + fold_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED + fold_idx)

    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx} | Train: {len(train_samples)} | Val: {len(val_samples)}")
    print(f"{'='*80}")

    train_ds_fold = FlatSampleDataset(train_samples, transform=train_tf)
    val_ds_fold   = FlatSampleDataset(val_samples,   transform=eval_tf)
    train_loader_fold = DataLoader(
        train_ds_fold, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )
    val_loader_fold = DataLoader(
        val_ds_fold, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )

    model     = MultiTaskSwinBase(num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                             pretrained=PRETRAINED, dropout=DROPOUT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")

    start_epoch       = 0
    best_metric       = 0.0
    epochs_no_improve = 0
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc_species": [], "val_acc_species": [],
        "train_acc_health":  [], "val_acc_health":  [],
        "train_acc_both":    [], "val_acc_both":    []
    }

    resume_ckpt = find_latest_epoch_ckpt(fold_idx)
    if resume_ckpt is not None:
        print(f"Resuming from checkpoint: {resume_ckpt}")
        ckpt = torch.load(resume_ckpt, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch       = ckpt["epoch"] + 1
        best_metric       = ckpt["best_metric"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history           = ckpt["history"]
        print(f"Resumed from epoch {ckpt['epoch']+1}, best_metric={best_metric:.4f}")

    for epoch in range(start_epoch, EPOCHS):
        print(f"\nFold {fold_idx} | Epoch {epoch+1}/{EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        train_stats = run_epoch(train_loader_fold, model, optimizer, train=True,  epoch=epoch)
        val_stats   = run_epoch(val_loader_fold,   model, optimizer=None, train=False, epoch=epoch)
        scheduler.step()

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_acc_species"].append(train_stats["acc_species"])
        history["val_acc_species"].append(val_stats["acc_species"])
        history["train_acc_health"].append(train_stats["acc_health"])
        history["val_acc_health"].append(val_stats["acc_health"])
        history["train_acc_both"].append(train_stats["acc_both"])
        history["val_acc_both"].append(val_stats["acc_both"])

        print(f"  Train - Loss: {train_stats['loss']:.4f} | Sp: {train_stats['acc_species']:.3f} | He: {train_stats['acc_health']:.3f}")
        print(f"  Val   - Loss: {val_stats['loss']:.4f} | Sp: {val_stats['acc_species']:.3f} | He: {val_stats['acc_health']:.3f}")

        val_metric = (val_stats["acc_species"] + val_stats["acc_health"]) / 2.0

        if val_metric > best_metric:
            best_metric = val_metric
            epochs_no_improve = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_metric": best_metric},
                       f"fold_{fold_idx}_best.pt")
            print(f"  ★ New best (val_metric={best_metric:.4f})")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")

        prev_ckpt = find_latest_epoch_ckpt(fold_idx)
        epoch_ckpt_path = f"fold_{fold_idx}_epoch_{epoch}.pt"
        torch.save({
            "model":             model.state_dict(),
            "optimizer":         optimizer.state_dict(),
            "scheduler":         scheduler.state_dict(),
            "epoch":             epoch,
            "best_metric":       best_metric,
            "epochs_no_improve": epochs_no_improve,
            "history":           history
        }, epoch_ckpt_path)
        if prev_ckpt is not None and prev_ckpt != epoch_ckpt_path:
            try:
                os.remove(prev_ckpt)
            except FileNotFoundError:
                pass

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping after {PATIENCE} epochs without improvement.")
            break

    best_ckpt = torch.load(f"fold_{fold_idx}_best.pt", map_location=device)
    model.load_state_dict(best_ckpt["model"])
    model.eval()
    all_sp_preds, all_sp_true, all_he_preds, all_he_true = [], [], [], []
    with torch.no_grad():
        for imgs, y_sp, y_he in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            logits_sp, logits_he = model(imgs)
            all_sp_preds.extend(logits_sp.argmax(1).cpu().numpy())
            all_sp_true.extend(y_sp.numpy())
            all_he_preds.extend(logits_he.argmax(1).cpu().numpy())
            all_he_true.extend(y_he.numpy())

    task_a_acc = accuracy_score(all_sp_true, all_sp_preds)
    task_b_acc = accuracy_score(all_he_true, all_he_preds)
    print(f"\nFold {fold_idx} Test — Species: {task_a_acc:.4f} | Health: {task_b_acc:.4f}")

    pd.DataFrame(history).to_csv(f"fold_{fold_idx}_history.csv", index=False)

    return {"task_a_accuracy": float(task_a_acc), "task_b_accuracy": float(task_b_acc)}

## COMPREHENSIVE TESTING FUNCTION

In [ ]:
def comprehensive_test(model, test_loader, device, species_map, health_map):
    """
    Perform comprehensive testing with metrics and visualizations
    """
    model.eval()
    
    # Storage for predictions and ground truth
    all_species_preds = []
    all_species_true = []
    all_health_preds = []
    all_health_true = []
    all_both_correct = []
    
    print("Running comprehensive test evaluation...")
    
    with torch.no_grad():
        for batch_idx, (imgs, y_species, y_health) in enumerate(test_loader):
            imgs = imgs.to(device, non_blocking=True)
            y_species = y_species.to(device, non_blocking=True)
            y_health = y_health.to(device, non_blocking=True)
            
            # Get predictions
            logits_species, logits_health = model(imgs)
            preds_species = logits_species.argmax(dim=1)
            preds_health = logits_health.argmax(dim=1)
            
            # Store predictions and ground truth
            all_species_preds.extend(preds_species.cpu().numpy())
            all_species_true.extend(y_species.cpu().numpy())
            all_health_preds.extend(preds_health.cpu().numpy())
            all_health_true.extend(y_health.cpu().numpy())
            
            # Check if both predictions are correct
            both_correct = ((preds_species == y_species) & (preds_health == y_health)).cpu().numpy()
            all_both_correct.extend(both_correct)
    
    # Convert to numpy arrays
    all_species_preds = np.array(all_species_preds)
    all_species_true = np.array(all_species_true)
    all_health_preds = np.array(all_health_preds)
    all_health_true = np.array(all_health_true)
    all_both_correct = np.array(all_both_correct)
    
    # Reverse mapping for labels
    species_labels = {v: k.capitalize() for k, v in species_map.items()}
    health_labels = {v: k.capitalize() for k, v in health_map.items()}
    
    # -------------------------------
    # Print Metrics
    # -------------------------------
    print("\n" + "="*80)
    print("COMPREHENSIVE TEST RESULTS")
    print("="*80)
    
    # Overall accuracies
    species_acc = accuracy_score(all_species_true, all_species_preds)
    health_acc = accuracy_score(all_health_true, all_health_preds)
    both_acc = all_both_correct.mean()
    
    print(f"\n{'OVERALL ACCURACIES':^80}")
    print("-"*80)
    print(f"  Species Classification:  {species_acc:.4f} ({species_acc*100:.2f}%)")
    print(f"  Health Classification:   {health_acc:.4f} ({health_acc*100:.2f}%)")
    print(f"  Both Correct (Joint):    {both_acc:.4f} ({both_acc*100:.2f}%)")
    
    # Species Classification Report
    print(f"\n{'SPECIES CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_species_true,
        all_species_preds,
        target_names=[species_labels[i] for i in sorted(species_labels.keys())],
        digits=4
    ))
    
    # Health Classification Report
    print(f"\n{'HEALTH/DISEASE CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_health_true,
        all_health_preds,
        target_names=[health_labels[i] for i in sorted(health_labels.keys())],
        digits=4
    ))
    
    # Per-class joint accuracy
    print(f"\n{'PER-CLASS JOINT ACCURACY':^80}")
    print("-"*80)
    for sp_id in sorted(species_labels.keys()):
        for he_id in sorted(health_labels.keys()):
            # Find samples of this joint class
            mask = (all_species_true == sp_id) & (all_health_true == he_id)
            if mask.sum() > 0:
                joint_acc = all_both_correct[mask].mean()
                count = mask.sum()
                sp_name = species_labels[sp_id]
                he_name = health_labels[he_id]
                print(f"  {sp_name:8s} + {he_name:12s}: {joint_acc:.4f} "
                      f"({joint_acc*100:.2f}%) [{count:4d} samples]")
    
    # -------------------------------
    # Visualizations
    # -------------------------------
    
    # 1. Species Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_species = confusion_matrix(all_species_true, all_species_preds)
    sns.heatmap(
        cm_species,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        xticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
        yticklabels=[species_labels[i] for i in sorted(species_labels.keys())]
    )
    ax.set_title(f'Species Classification\nAccuracy: {species_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_species.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"\nSaved species confusion matrix to 'confusion_matrix_species.png'")
    plt.close()
    
    # 2. Health Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_health = confusion_matrix(all_health_true, all_health_preds)
    sns.heatmap(
        cm_health,
        annot=True,
        fmt='d',
        cmap='Greens',
        ax=ax,
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[health_labels[i] for i in sorted(health_labels.keys())]
    )
    ax.set_title(f'Health/Disease Classification\nAccuracy: {health_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_health.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"Saved health confusion matrix to 'confusion_matrix_health.png'")
    plt.close()
    
    # 3. Joint Accuracy Heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    joint_acc_matrix = np.zeros((len(species_labels), len(health_labels)))
    
    for sp_id in sorted(species_labels.keys()):
        for he_id in sorted(health_labels.keys()):
            mask = (all_species_true == sp_id) & (all_health_true == he_id)
            if mask.sum() > 0:
                joint_acc_matrix[sp_id, he_id] = all_both_correct[mask].mean()
    
    sns.heatmap(
        joint_acc_matrix,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={'label': 'Accuracy'}
    )
    ax.set_title('Joint Classification Accuracy by Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Species')
    ax.set_xlabel('Health Status')
    plt.tight_layout()
    plt.savefig('joint_accuracy_heatmap.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"Saved joint accuracy heatmap to 'joint_accuracy_heatmap.png'")
    plt.close()
    
    print("\n" + "="*80)
    print("Testing complete! Generated visualizations:")
    print("  - confusion_matrix_species.png")
    print("  - confusion_matrix_health.png")
    print("  - joint_accuracy_heatmap.png")
    print("="*80 + "\n")
    
    return {
        'species_accuracy': species_acc,
        'health_accuracy': health_acc,
        'joint_accuracy': both_acc,
        'species_preds': all_species_preds,
        'species_true': all_species_true,
        'health_preds': all_health_preds,
        'health_true': all_health_true
    }

## K-FOLD CROSS VALIDATION

In [ ]:
def load_kfold_state():
    """Load k-fold training state from JSON file."""
    if Path(STATE_FILE).exists():
        with open(STATE_FILE, "r") as f:
            return json.load(f)
    return {"completed_folds": [], "fold_results": {}}

def save_kfold_state(state):
    """Save k-fold training state to JSON file."""
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_indices = list(range(len(all_samples)))

state = load_kfold_state()
print(f"K-fold state: completed folds = {state['completed_folds']}")
print(f"Starting {N_FOLDS}-fold stratified cross-validation...\n")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(all_indices, strat_labels)):
    if fold_idx in state["completed_folds"]:
        print(f"Skipping fold {fold_idx} (already completed)")
        continue

    train_samples_fold = [all_samples[i] for i in train_idx]
    val_samples_fold   = [all_samples[i] for i in val_idx]

    fold_result = train_fold(fold_idx, train_samples_fold, val_samples_fold)

    state["completed_folds"].append(fold_idx)
    state["fold_results"][str(fold_idx)] = fold_result
    save_kfold_state(state)
    print(f"\nFold {fold_idx} completed and saved: {fold_result}")

print("\n" + "="*80)
print("ALL FOLDS COMPLETE")
print("="*80)

## COMPREHENSIVE TEST (BEST FOLD)

In [ ]:
state = load_kfold_state()
fold_results = state["fold_results"]

best_fold_idx = int(max(fold_results.keys(),
                        key=lambda k: fold_results[k]["task_a_accuracy"]))
print(f"Best fold: {best_fold_idx} "
      f"(species_acc={fold_results[str(best_fold_idx)]['task_a_accuracy']:.4f})")

best_model = MultiTaskSwinBase(num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                          pretrained=PRETRAINED, dropout=DROPOUT).to(device)
ckpt = torch.load(f"fold_{best_fold_idx}_best.pt", map_location=device)
best_model.load_state_dict(ckpt["model"])
print(f"Loaded fold_{best_fold_idx}_best.pt")

print("\n" + "="*80)
print("Running Comprehensive Test Evaluation")
print("="*80 + "\n")

test_results = comprehensive_test(
    model=best_model, test_loader=test_loader, device=device,
    species_map=SPECIES_MAP, health_map=HEALTH_MAP
)

## SAMPLE PREDICTIONS (BEST FOLD)

In [ ]:
# Rebuild val loader for best fold using same StratifiedKFold split
skf_vis = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits_vis = list(skf_vis.split(list(range(len(all_samples))), strat_labels))
_, val_idx_best = fold_splits_vis[best_fold_idx]
val_samples_best = [all_samples[i] for i in val_idx_best]
val_ds_best = FlatSampleDataset(val_samples_best, transform=eval_tf)
val_loader_best = DataLoader(
    val_ds_best, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

print("\n" + "="*80)
print("Generating Sample Predictions Visualization")
print("="*80 + "\n")

# Label mappings for visualization
species_labels = {
    0: 'Guava',
    1: 'Mango',
    2: 'Papaya'
}

health_labels = {
    0: 'Healthy',
    1: 'Anthracnose',
}

# Number of samples to display per class
amount = 10

# Collect samples from validation set
sample_images_by_class = {0: [], 1: [], 2: []}
sample_predictions_by_class = {0: [], 1: [], 2: []}
sample_ground_truth_by_class = {0: [], 1: [], 2: []}

best_model.eval()
with torch.no_grad():
    for images, species_batch, health_batch in val_loader_best:
        images = images.to(device)
        
        outputs = best_model(images)
        species_preds = outputs[0].argmax(1)
        health_preds = outputs[1].argmax(1)
        
        for i in range(len(images)):
            species_class = species_batch[i].item()
            
            if len(sample_images_by_class[species_class]) < amount:
                sample_images_by_class[species_class].append(images[i].cpu())
                sample_predictions_by_class[species_class].append({
                    'species': species_preds[i].item(),
                    'health': health_preds[i].item()
                })
                sample_ground_truth_by_class[species_class].append({
                    'species': species_batch[i].item(),
                    'health': health_batch[i].item()
                })
        
        if all(len(samples) >= amount for samples in sample_images_by_class.values()):
            break

# Visualize
mean = np.array(Config.IMG_MEAN)
std = np.array(Config.IMG_STD)

fig, axes = plt.subplots(NUM_SPECIES, amount, figsize=(3*amount, 9))

for row, species_idx in enumerate(sorted(sample_images_by_class.keys())):
    for col in range(amount):
        ax = axes[row, col]
        
        img = sample_images_by_class[species_idx][col]
        pred = sample_predictions_by_class[species_idx][col]
        gt = sample_ground_truth_by_class[species_idx][col]
        
        # Denormalize and display
        img_display = img.numpy().transpose(1, 2, 0)
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1)
        
        ax.imshow(img_display)
        ax.axis('off')
        
        # Check correctness
        both_correct = (pred['species'] == gt['species']) and (pred['health'] == gt['health'])
        
        # Create title
        pred_sp = species_labels[pred['species']]
        pred_he = health_labels[pred['health']]
        gt_sp = species_labels[gt['species']]
        gt_he = health_labels[gt['health']]
        
        title = f"Pred: {pred_sp}, {pred_he}\nTrue: {gt_sp}, {gt_he}"
        color = 'green' if both_correct else 'red'
        ax.set_title(title, fontsize=8, color=color, fontweight='bold')
    
    # Add species label
    fig.text(0.02, 0.5 + (1 - row) * 0.3, species_labels[species_idx],
             fontsize=12, fontweight='bold', va='center', rotation=90)

plt.suptitle(f'Sample Predictions (Best Fold {best_fold_idx}) - {amount} Samples per Class\n(Green=Correct, Red=Incorrect)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0.05, 0, 1, 0.99])
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
print(f"Saved sample predictions ({NUM_SPECIES*amount} total: {amount} per class) to 'sample_predictions.png'")
plt.close()


## FINAL SUMMARY

In [ ]:
state = load_kfold_state()
fold_results = state["fold_results"]

print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS SUMMARY")
print("="*80)
print(f"\n{'Fold':>6} | {'Species Acc (Task A)':>22} | {'Health Acc (Task B)':>22}")
print("-" * 58)

task_a_accs, task_b_accs = [], []
for fold_idx in range(N_FOLDS):
    key = str(fold_idx)
    if key in fold_results:
        ta = fold_results[key]["task_a_accuracy"]
        tb = fold_results[key]["task_b_accuracy"]
        task_a_accs.append(ta)
        task_b_accs.append(tb)
        print(f"{fold_idx:>6} | {ta*100:>20.2f}% | {tb*100:>20.2f}%")

print("-" * 58)
print(f"\nTask A Accuracy : {np.mean(task_a_accs)*100:.2f} \u00b1 {np.std(task_a_accs)*100:.2f}")
print(f"Task B Accuracy : {np.mean(task_b_accs)*100:.2f} \u00b1 {np.std(task_b_accs)*100:.2f}")
print("="*80)